# Optical-Only v14 Time Embedding And Attention Plots

这个 notebook 做两件事：
1. 读取 `optical_only_kn_v14` checkpoint，绘制 `mTAN` 的时间嵌入函数 `phi(t)`。
2. 从训练集随机抽取一条千新星光变曲线，按开头硬编码的时间窗口裁剪后，绘制“光变曲线 + 时间注意力热力图”示例。

说明：
- 不接入推理脚本主流程，只做独立分析。
- 时间注意力图的样式尽量与真实数据推理结果保持一致。
- 时间窗口通过 `TIME_WINDOW_DAYS` 在开头硬编码。


In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

import h5py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.lines import Line2D
from torch.amp import autocast

CKPT_PATH = Path('<BASE_DIR>/data/model/checkpoints_optical/optical_only/optical_only_kn_v15_win1020/optical_only_best.pth')
TRAIN_H5_PATH = Path('<BASE_DIR>/data/Optical_Only_dataset/combined_dataset_train.h5')
INFER_SCRIPT_PATH = Path('<BASE_DIR>/fink-lsst/optical_only_kn_lsst_inference.py')
OUTPUT_DIR = Path('<BASE_DIR>/gw-kn-multimodal/optical_only/outputs/plots')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_WINDOW_DAYS = (-10.0, 20.0)
TIME_WINDOW_SCALED = tuple(day / 100.0 for day in TIME_WINDOW_DAYS)
MAX_SAMPLE_SELECTION_TRIES = 256

PSFFLUX_ZP = 31.4
ASINH_MAG_FACTOR = 2.5 / np.log(10.0)
BAND_COLORS = {
    'u': '#6d6d6d',
    'g': '#2ca02c',
    'r': '#d62728',
    'i': '#ff7f0e',
    'z': '#8c564b',
    'y': '#9467bd',
}
SEED = 42
rng = np.random.default_rng(SEED)


## 1. 读取 checkpoint 并绘制头平均的 `phi(t)`

这里先对全部 attention heads 求平均，再在参考时间窗口 `[-0.3, 0.6]` 上计算 `phi(t)`。


In [ ]:
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
prefix = 'optical_encoder.time_embedding.'

w0 = state[prefix + 'w0'].detach().cpu().numpy().reshape(-1, 1)
a0 = state[prefix + 'a0'].detach().cpu().numpy().reshape(-1, 1)
wi = state[prefix + 'wi'].detach().cpu().numpy().reshape(w0.shape[0], -1)
ai = state[prefix + 'ai'].detach().cpu().numpy().reshape(w0.shape[0], -1)

print({'num_heads': int(w0.shape[0]), 'ref_dim': int(wi.shape[1] + 1)})

t_vals = np.linspace(-0.1, 0.2, 512, dtype=np.float64)
t = t_vals.reshape(1, -1, 1)
linear = w0[:, None, :] * t + a0[:, None, :]
periodic = np.sin(wi[:, None, :] * t + ai[:, None, :])
phi = np.concatenate([linear, periodic], axis=-1)  # [H, T, 64]
phi_head_avg = phi.mean(axis=0)

phi_df = pd.DataFrame(phi_head_avg, columns=[f'dim_{i:03d}' for i in range(phi_head_avg.shape[1])])
phi_df.insert(0, 't_scaled', t_vals)
phi_df.insert(1, 't_days', t_vals * 100.0)
phi_csv = OUTPUT_DIR / 'mtan_phi_headavg.csv'
phi_png = OUTPUT_DIR / 'mtan_phi_headavg.png'
phi_df.to_csv(phi_csv, index=False)

highlight_dims = min(8, phi_head_avg.shape[1])
fig, ax = plt.subplots(figsize=(11, 6))
for dim_idx in range(phi_head_avg.shape[1]):
    if dim_idx < highlight_dims:
        ax.plot(t_vals, phi_head_avg[:, dim_idx], linewidth=2.0, label=f'dim {dim_idx}')
    else:
        ax.plot(t_vals, phi_head_avg[:, dim_idx], color='#bdbdbd', alpha=0.18, linewidth=0.8)
ax.axhline(0.0, color='#333333', linewidth=0.8, alpha=0.6)
ax.set_title('Head-averaged mTAN phi(t) over reference window [-0.1, 0.2] (scaled units)')
ax.set_xlabel('t (scaled units)')
ax.set_ylabel('phi(t)')
ax.grid(alpha=0.22, linestyle='--')
ax.legend(loc='upper left', ncol=2, frameon=False)
fig.tight_layout()
fig.savefig(phi_png, dpi=180, bbox_inches='tight')
plt.show()
print({'phi_png': str(phi_png), 'phi_csv': str(phi_csv)})


## 1.1 绘制全部 heads 的前若干维时间嵌入函数

这一节不对 head 求平均，而是把每个 head 单独展开，并绘制每个 head 的前 `num_plot_dims` 个时间嵌入维度曲线。


In [ ]:
num_plot_dims = 4
num_heads = int(w0.shape[0])
phi_all_heads_png = OUTPUT_DIR / f'mtan_phi_all_heads_first{num_plot_dims}dims.png'
phi_all_heads_csv = OUTPUT_DIR / f'mtan_phi_all_heads_first{num_plot_dims}dims.csv'

rows = []
for head_idx in range(num_heads):
    for dim_idx in range(min(num_plot_dims, phi.shape[2])):
        rows.append(pd.DataFrame({
            "head": head_idx,
            "dim": dim_idx,
            "t_scaled": t_vals,
            "t_days": t_vals * 100.0,
            "phi": phi[head_idx, :, dim_idx],
        }))
pd.concat(rows, ignore_index=True).to_csv(phi_all_heads_csv, index=False)

ncols = 2
nrows = int(np.ceil(num_heads / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.2 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']

for head_idx in range(num_heads):
    ax = axes[head_idx]
    for dim_idx in range(min(num_plot_dims, phi.shape[2])):
        ax.plot(t_vals, phi[head_idx, :, dim_idx], linewidth=1.8, color=colors[dim_idx % len(colors)], label=f'dim {dim_idx}')
    ax.axhline(0.0, color='#333333', linewidth=0.8, alpha=0.6)
    ax.set_title(f'Head {head_idx}')
    ax.set_ylabel('phi_h(t)')
    ax.grid(alpha=0.22, linestyle='--')
    ax.legend(loc='upper left', frameon=False, ncol=2, fontsize=9)

for ax in axes[num_heads:]:
    ax.set_axis_off()
for ax in axes[max(0, num_heads - ncols):num_heads]:
    ax.set_xlabel('t (scaled units)')

fig.suptitle(f'mTAN phi(t): all heads, first {num_plot_dims} dims', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(phi_all_heads_png, dpi=180, bbox_inches='tight')
plt.show()
print({'phi_all_heads_png': str(phi_all_heads_png), 'phi_all_heads_csv': str(phi_all_heads_csv)})


## 1.2 统计全部 heads 的全部时间嵌入维度周期

对周期项 `sin(w_i t + a_i)`，周期按 `T = 2\pi / |w_i|` 计算。这里同时导出长表 CSV、`head × dim` 周期矩阵 CSV，并绘制一个按 `log10(period_days)` 着色的热力图。注意 `dim_000` 是线性项 `w_0 t + a_0`，没有有限周期，因此表里记为 `NaN`。


In [ ]:
period_eps = 1.0e-12
num_dims = int(phi.shape[2])
abs_wi = np.abs(wi)

period_scaled = np.full((num_heads, num_dims), np.nan, dtype=np.float64)
period_days = np.full((num_heads, num_dims), np.nan, dtype=np.float64)
period_scaled[:, 1:] = np.where(abs_wi > period_eps, 2.0 * np.pi / abs_wi, np.nan)
period_days[:, 1:] = period_scaled[:, 1:] * 100.0

omega_scaled = np.full((num_heads, num_dims), np.nan, dtype=np.float64)
omega_scaled[:, 1:] = abs_wi

period_long_df = pd.DataFrame({
    'head': np.repeat(np.arange(num_heads, dtype=np.int64), num_dims),
    'dim': np.tile(np.arange(num_dims, dtype=np.int64), num_heads),
    'component': np.where(np.tile(np.arange(num_dims), num_heads) == 0, 'linear', 'periodic'),
    'omega_scaled': omega_scaled.reshape(-1),
    'period_scaled': period_scaled.reshape(-1),
    'period_days': period_days.reshape(-1),
})
period_long_df['dim_name'] = period_long_df['dim'].map(lambda d: f'dim_{d:03d}')

period_matrix_days_df = pd.DataFrame(
    period_days,
    index=[f'head_{head_idx:02d}' for head_idx in range(num_heads)],
    columns=[f'dim_{dim_idx:03d}' for dim_idx in range(num_dims)],
)

period_long_csv = OUTPUT_DIR / 'mtan_time_embedding_periods_all_heads_long.csv'
period_matrix_csv = OUTPUT_DIR / 'mtan_time_embedding_periods_all_heads_matrix_days.csv'
period_heatmap_png = OUTPUT_DIR / 'mtan_time_embedding_periods_all_heads_heatmap.png'
period_long_df.to_csv(period_long_csv, index=False)
period_matrix_days_df.to_csv(period_matrix_csv)

heat_data = np.ma.masked_invalid(np.log10(period_days[:, 1:]))
fig, ax = plt.subplots(figsize=(14, 4.8))
im = ax.imshow(heat_data, aspect='auto', cmap='viridis')
xticks = np.arange(0, num_dims - 1, max(1, (num_dims - 1) // 16))
ax.set_xticks(xticks)
ax.set_xticklabels([f'dim_{dim_idx + 1:03d}' for dim_idx in xticks], rotation=45, ha='right')
ax.set_yticks(np.arange(num_heads))
ax.set_yticklabels([f'head {head_idx}' for head_idx in range(num_heads)])
ax.set_title('mTAN time-embedding periods for all heads and periodic dims')
ax.set_xlabel('embedding dimension (periodic terms only)')
ax.set_ylabel('attention head')
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
cb.set_label('log10(period [days])')
fig.tight_layout()
fig.savefig(period_heatmap_png, dpi=180, bbox_inches='tight')
plt.show()

finite_period_days = period_days[:, 1:][np.isfinite(period_days[:, 1:])]
print(period_long_df.head(12).to_string(index=False))
print({
    'period_long_csv': str(period_long_csv),
    'period_matrix_csv': str(period_matrix_csv),
    'period_heatmap_png': str(period_heatmap_png),
    'period_days_min': float(finite_period_days.min()),
    'period_days_max': float(finite_period_days.max()),
})


## 2. 从训练集随机抽取一条千新星光变，并绘制“光变曲线 + 时间注意力”

下面的单元会：
- 从 `combined_dataset_train.h5` 随机抽取一条正样本光变曲线
- 按 `TIME_WINDOW_DAYS` 指定的相对时间窗口裁剪观测点
- 加载当前 `v14` 模型
- 计算 5 个 offset 平均后的时间注意力热力图
- 生成一张与真实数据推理结果风格接近的组合图


In [ ]:
def load_infer_module(script_path: Path):
    spec = importlib.util.spec_from_file_location('fink_optical_infer_mod', script_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod


def luptitude_to_flux_and_sigma(values_lupt: np.ndarray, errors_lupt: np.ndarray, band_index: np.ndarray, lupt_b_njy: np.ndarray):
    b = lupt_b_njy[band_index]
    two_b = 2.0 * b
    x = (PSFFLUX_ZP - values_lupt) / ASINH_MAG_FACTOR - np.log(b)
    flux = two_b * np.sinh(x)
    fluxerr = np.abs(errors_lupt) * np.sqrt(np.square(flux) + np.square(two_b)) / ASINH_MAG_FACTOR
    return flux, fluxerr


def flux_to_mag(flux_njy: np.ndarray, fluxerr_njy: np.ndarray):
    valid = np.isfinite(flux_njy) & np.isfinite(fluxerr_njy) & (flux_njy > 0) & (fluxerr_njy > 0)
    mag = np.full_like(flux_njy, np.nan, dtype=np.float64)
    magerr = np.full_like(flux_njy, np.nan, dtype=np.float64)
    if np.any(valid):
        mag[valid] = PSFFLUX_ZP - 2.5 * np.log10(flux_njy[valid])
        magerr[valid] = (2.5 / np.log(10.0)) * (fluxerr_njy[valid] / flux_njy[valid])
    return mag, magerr, valid


def plot_status_errorbars(ax, x, y, yerr, is_detection, color):
    det = np.asarray(is_detection, dtype=bool)
    plotted = False
    if np.any(det):
        ax.errorbar(x[det], y[det], yerr=yerr[det], fmt='o', markersize=4.5, color=color, ecolor=color, elinewidth=1.0, capsize=0, alpha=0.95)
        plotted = True
    if np.any(~det):
        ax.errorbar(x[~det], y[~det], yerr=yerr[~det], fmt='s', markersize=4.0, mfc='white', mec=color, color=color, ecolor=color, elinewidth=0.9, capsize=0, alpha=0.8)
        plotted = True
    return plotted


def crop_sample_to_time_window(sample: dict[str, object], time_window_scaled: tuple[float, float]):
    start_scaled, end_scaled = map(float, time_window_scaled)
    times = np.asarray(sample['times'], dtype=np.float32)
    masks = np.asarray(sample['masks'], dtype=np.float32)
    slot_valid_full = masks.sum(axis=1) > 0
    window_mask = np.isfinite(times) & (times >= start_scaled) & (times <= end_scaled)
    keep_mask = slot_valid_full & window_mask
    if not np.any(keep_mask):
        return None
    cropped = dict(sample)
    cropped['n_points_full'] = int(slot_valid_full.sum())
    cropped['n_points_window'] = int(keep_mask.sum())
    cropped['time_window_days'] = tuple(float(x) for x in TIME_WINDOW_DAYS)
    cropped['slot_keep_mask'] = keep_mask
    for key in ('times', 'values', 'errors', 'masks', 'slot_is_detection'):
        cropped[key] = np.asarray(sample[key])[keep_mask]
    cropped['n_det_window'] = int(np.count_nonzero(np.asarray(cropped['slot_is_detection']) > 0.5))
    cropped['n_bands_window'] = int(np.count_nonzero(np.asarray(cropped['masks']).sum(axis=0) > 0))
    return cropped


with h5py.File(TRAIN_H5_PATH, 'r') as f:
    grp = f['events/optical_data']
    n_samples = int(grp['times'].shape[0])
    lupt_b_njy = np.asarray(f.attrs['lupt_b_njy'], dtype=np.float64)
    band_order = [b.strip().lower() for b in str(f.attrs.get('lupt_band_order', 'u,g,r,i,z,Y')).split(',')]
    sample = None
    for _ in range(MAX_SAMPLE_SELECTION_TRIES):
        sample_index = int(rng.integers(0, n_samples))
        candidate = {
            'sample_index': sample_index,
            'times': np.asarray(grp['times'][sample_index], dtype=np.float32),
            'values': np.asarray(grp['values'][sample_index], dtype=np.float32),
            'errors': np.asarray(grp['errors'][sample_index], dtype=np.float32),
            'masks': np.asarray(grp['masks'][sample_index], dtype=np.float32),
            'slot_is_detection': np.asarray(grp['slot_is_detection'][sample_index], dtype=np.float32),
            'zero_time_mjd_base': float(np.asarray(grp['zero_time_mjd_base'][sample_index], dtype=np.float64)),
            'meta_n_det': int(np.asarray(grp['meta_n_det'][sample_index]).item()),
            'meta_n_bands': int(np.asarray(grp['meta_n_bands'][sample_index]).item()),
            'lupt_b_njy': lupt_b_njy,
            'band_order': band_order,
        }
        sample = crop_sample_to_time_window(candidate, TIME_WINDOW_SCALED)
        if sample is not None:
            break
    if sample is None:
        raise RuntimeError(f'Failed to find a sample with valid observations inside TIME_WINDOW_DAYS={TIME_WINDOW_DAYS} after {MAX_SAMPLE_SELECTION_TRIES} tries.')

slot_valid = sample['masks'].sum(axis=1) > 0
slot_indices = np.flatnonzero(slot_valid)
rows = []
for slot_idx in slot_indices:
    band_idx = int(np.argmax(sample['masks'][slot_idx]))
    band = sample['band_order'][band_idx]
    t_scaled = float(sample['times'][slot_idx])
    mjd = sample['zero_time_mjd_base'] + 100.0 * t_scaled
    lupt = float(sample['values'][slot_idx, band_idx])
    lupt_err = float(abs(sample['errors'][slot_idx, band_idx]))
    flux_njy, fluxerr_njy = luptitude_to_flux_and_sigma(
        np.asarray([lupt], dtype=np.float64),
        np.asarray([lupt_err], dtype=np.float64),
        np.asarray([band_idx], dtype=np.int64),
        sample['lupt_b_njy'],
    )
    mag, magerr, mag_valid = flux_to_mag(flux_njy, fluxerr_njy)
    rows.append({
        'slot_idx': int(slot_idx),
        'band': band,
        'mjd': mjd,
        'lupt': lupt,
        'lupt_err': lupt_err,
        'mag': float(mag[0]) if mag_valid[0] else float('nan'),
        'magerr': float(magerr[0]) if mag_valid[0] else float('nan'),
        'is_detection': bool(sample['slot_is_detection'][slot_idx] > 0.5),
    })
lc_df = pd.DataFrame(rows)

infer_mod = load_infer_module(INFER_SCRIPT_PATH)
parser = infer_mod.build_parser()
args = parser.parse_args([])
config = infer_mod.build_config(args)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_dtype = config.amp_dtype(device)
model, _, _ = infer_mod.load_optical_only_model(config, device)
offset_days = infer_mod.load_offset_days(config.offset_npz, config.offset_key, config.offset_quantiles)

opt_t = torch.from_numpy(sample['times'][None, :]).to(device=device, dtype=torch.float32)
opt_v = torch.from_numpy(sample['values'][None, :, :]).to(device=device, dtype=torch.float32)
opt_mask = torch.from_numpy(sample['masks'][None, :, :]).to(device=device, dtype=torch.float32)
opt_err = torch.from_numpy(sample['errors'][None, :, :]).to(device=device, dtype=torch.float32)
ref_time = infer_mod.build_ref_time(1, config.n_ref, config.ref_start, config.ref_end, device, opt_t.dtype)
slot_valid_t = (opt_mask.sum(dim=-1) > 0)
band_present = (opt_mask.sum(dim=1) > 0)
shifted_t_stack = []
attn_norm_stack = []
with torch.no_grad():
    for off_days in np.asarray(offset_days, dtype=np.float64).tolist():
        delta_days = torch.full((1,), float(off_days), device=device, dtype=torch.float32)
        shifted_t = infer_mod.apply_time_offsets(opt_t, opt_mask, delta_days=delta_days, scale_divisor=config.offset_scale_days_divisor)
        with autocast(device_type='cuda', dtype=amp_dtype, enabled=(device.type == 'cuda')):
            _, _, attn_weights = model.encode_optical_with_attention(shifted_t, opt_v, ref_time, opt_mask, opt_err)
        attn_ref = attn_weights.float()[:, :, 1:, :, :]
        attn_head_mean = attn_ref.mean(dim=1)
        band_present_f = band_present[:, None, None, :].to(dtype=attn_head_mean.dtype)
        attn_band_sum = (attn_head_mean * band_present_f).sum(dim=-1)
        slot_valid_f = slot_valid_t[:, None, :].to(dtype=attn_band_sum.dtype)
        attn_band_sum = attn_band_sum * slot_valid_f
        row_sum = attn_band_sum.sum(dim=-1, keepdim=True)
        attn_norm = torch.where(row_sum > 0, attn_band_sum / row_sum, torch.zeros_like(attn_band_sum))
        shifted_t_stack.append(shifted_t.detach().cpu().numpy()[0])
        attn_norm_stack.append(attn_norm.detach().cpu().numpy()[0])

shifted_t_stack = np.asarray(shifted_t_stack, dtype=np.float64)
attn_norm_stack = np.asarray(attn_norm_stack, dtype=np.float64)
attn_mean = attn_norm_stack.mean(axis=0)
shifted_mean = shifted_t_stack.mean(axis=0)
slot_valid_np = (sample['masks'].sum(axis=1) > 0)
slot_band_idx = np.argmax(sample['masks'][slot_valid_np], axis=1)
idx_to_band = {i: b for i, b in enumerate(sample['band_order'])}
slot_band_labels = np.asarray([idx_to_band[int(i)] for i in slot_band_idx], dtype=object)
heat = attn_mean[:, slot_valid_np]
obs_time_mean_days = shifted_mean[slot_valid_np] * 100.0
ref_axis_days = ref_time[0].detach().cpu().numpy().astype(np.float64) * 100.0

plot_png = OUTPUT_DIR / 'training_kn_lightcurve_attention_example.png'
meta_json = OUTPUT_DIR / 'training_kn_lightcurve_attention_example.json'
fig = plt.figure(figsize=(12, 10))
gs = fig.add_gridspec(3, 1, height_ratios=[1.0, 1.0, 1.25], hspace=0.16)
ax_lupt = fig.add_subplot(gs[0, 0])
ax_mag = fig.add_subplot(gs[1, 0], sharex=ax_lupt)
ax_attn = fig.add_subplot(gs[2, 0])

for band, grp in lc_df.groupby('band', sort=False):
    color = BAND_COLORS.get(str(band).lower().replace('y', 'y'), '#333333')
    x = grp['mjd'].to_numpy(dtype=np.float64)
    plot_status_errorbars(ax_lupt, x, grp['lupt'].to_numpy(dtype=np.float64), grp['lupt_err'].to_numpy(dtype=np.float64), grp['is_detection'].to_numpy(dtype=bool), color)
    mag_valid = np.isfinite(grp['mag'].to_numpy(dtype=np.float64)) & np.isfinite(grp['magerr'].to_numpy(dtype=np.float64))
    if np.any(mag_valid):
        sub = grp.loc[mag_valid]
        plot_status_errorbars(ax_mag, sub['mjd'].to_numpy(dtype=np.float64), sub['mag'].to_numpy(dtype=np.float64), sub['magerr'].to_numpy(dtype=np.float64), sub['is_detection'].to_numpy(dtype=bool), color)

window_start_days, window_end_days = sample['time_window_days']
ax_lupt.set_title(f"Random training KN sample | idx={sample['sample_index']} | window=[{window_start_days:+.1f}, {window_end_days:+.1f}] d | n_det={sample['n_det_window']}/{sample['meta_n_det']} | n_bands={sample['n_bands_window']}/{sample['meta_n_bands']}")
ax_lupt.set_ylabel('luptitude')
ax_lupt.grid(alpha=0.25, linestyle='--')
ax_lupt.invert_yaxis()
ax_mag.set_xlabel('MJD')
ax_mag.set_ylabel('AB magnitude\n(positive flux only)')
ax_mag.grid(alpha=0.25, linestyle='--')
ax_mag.invert_yaxis()

band_handles = [Line2D([0], [0], color=BAND_COLORS[b], marker='o', linestyle='None', label=b) for b in ['u', 'g', 'r', 'i', 'z', 'y']]
status_handles = [
    Line2D([0], [0], color='#333333', marker='o', linestyle='None', label='detection'),
    Line2D([0], [0], color='#333333', marker='s', mfc='white', linestyle='None', label='non-detection/forced'),
]
ax_lupt.legend(handles=band_handles + status_handles, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)

obs_point_idx = np.arange(1, heat.shape[1] + 1, dtype=np.float64)
obs_edges = np.arange(0.5, heat.shape[1] + 1.5, 1.0, dtype=np.float64)
if len(ref_axis_days) == 1:
    ref_edges = np.asarray([ref_axis_days[0] - 0.5, ref_axis_days[0] + 0.5], dtype=np.float64)
else:
    mids = 0.5 * (ref_axis_days[:-1] + ref_axis_days[1:])
    ref_edges = np.concatenate([[ref_axis_days[0] - (mids[0] - ref_axis_days[0])], mids, [ref_axis_days[-1] + (ref_axis_days[-1] - mids[-1])]])
vmax = max(float(np.nanpercentile(heat, 99)), 1.0e-6)
mesh = ax_attn.pcolormesh(obs_edges, ref_edges, heat, shading='auto', cmap='magma', vmin=0.0, vmax=vmax)
ax_attn.set_title(f"Time attention | avg {len(offset_days)} offsets | mean over heads")
ax_attn.set_xlabel('observation point + band (5-offset mean relative time shown below)')
ax_attn.set_ylabel('reference time [days]')
ax_attn.set_xticks(obs_point_idx)
ax_attn.set_xticklabels([f"{idx}-{band}\n{time_mean:+.2f}d" for idx, band, time_mean in zip(obs_point_idx.astype(int), slot_band_labels, obs_time_mean_days)], fontsize=8)
ax_attn.tick_params(axis='x', rotation=0)
ax_attn.set_xlim(0.5, heat.shape[1] + 0.5)
ax_attn.set_xticks(obs_edges, minor=True)
ax_attn.grid(which='minor', axis='x', alpha=0.18, linestyle='-', linewidth=0.6)
ax_attn.grid(which='major', axis='y', alpha=0.12, linestyle='--', linewidth=0.7)
cb = fig.colorbar(mesh, ax=ax_attn, fraction=0.046, pad=0.02)
cb.set_label('attention weight (row-normalized over obs time, averaged over offsets)')

fig.tight_layout(rect=[0, 0, 0.86, 1])
fig.savefig(plot_png, dpi=180, bbox_inches='tight')
plt.show()
meta = {
    'sample_index': int(sample['sample_index']),
    'n_samples': int(n_samples),
    'n_det_full': int(sample['meta_n_det']),
    'n_bands_full': int(sample['meta_n_bands']),
    'n_points_full': int(sample['n_points_full']),
    'n_det_window': int(sample['n_det_window']),
    'n_bands_window': int(sample['n_bands_window']),
    'n_points_window': int(sample['n_points_window']),
    'time_window_days': [float(x) for x in sample['time_window_days']],
    'zero_time_mjd_base': float(sample['zero_time_mjd_base']),
    'offset_days': [float(x) for x in offset_days.tolist()],
    'output_png': str(plot_png),
}
meta_json.write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
